In [1]:
!pip install selenium webdriver-manager bs4 pandas

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time

def get_car_data(page):
    url = f"https://www.cardekho.com/used-cars+in+india-page{page}.htm"
    driver.get(url)
    time.sleep(3)  # wait for JS to load

    soup = BeautifulSoup(driver.page_source, "html.parser")

    cars = soup.find_all("div", class_="gsc_col-xs-12 gsc_col-sm-12 gsc_col-md-12 gsc_col-lg-12 holder")

    data = []
    for car in cars:
        try:
            name = car.find("span", class_="title").text.strip()
        except:
            name = None

        try:
            price = car.find("span", class_="price").text.strip()
        except:
            price = None

        try:
            details = car.find_all("span", class_="dotlist")
            year = details[0].text.strip()
            km = details[1].text.strip()
            fuel = details[2].text.strip()
            trns = details[3].text.strip()
        except:
            year = km = fuel = trns = None

        try:
            location = car.find("span", class_="cityName").text.strip()
        except:
            location = None

        data.append({
            "Car Name": name,
            "Price": price,
            "Year": year,
            "KM Driven": km,
            "Fuel Type": fuel,
            "Transmission": trns,
            "Location": location
        })

    return data


# START BROWSER
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

final_data = []

# किती pages scrape करायच्या ते तू बदलू शकतो
for page in range(1, 6):
    print("Scraping page:", page)
    cars = get_car_data(page)
    final_data.extend(cars)

driver.quit()

df = pd.DataFrame(final_data)
df.to_csv("CarDekho_UsedCars.csv", index=False)

print("DONE! Total records scraped:", len(df))
print(df.head())


Scraping page: 1
Scraping page: 2
Scraping page: 3
Scraping page: 4
Scraping page: 5
DONE! Total records scraped: 0
Empty DataFrame
Columns: []
Index: []


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
import time
import pandas as pd

# ---------- SETUP ----------
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
driver = webdriver.Chrome(options=options)

url = "https://www.cardekho.com/used-cars+in+pune"
driver.get(url)

time.sleep(5)

# SCROLL TO LOAD ALL CARS
for i in range(15):
    driver.find_element(By.TAG_NAME, "body").send_keys(Keys.END)
    time.sleep(2)

cars = driver.find_elements(By.CSS_SELECTOR, "div.NewUcExCard")

data = []

for car in cars:
    try:
        title = car.find_element(By.CSS_SELECTOR, "h3.title a").text
    except:
        title = ""

    try:
        details = car.find_element(By.CSS_SELECTOR, ".dotsDetails").text
        # Example: "55,285 kms • Petrol • Automatic"
        parts = details.split("•")
        kms = parts[0].strip()
        fuel = parts[1].strip()
        transmission = parts[2].strip()
    except:
        kms = fuel = transmission = ""

    try:
        price = car.find_element(By.CSS_SELECTOR, ".Price p").text
    except:
        price = ""

    try:
        old_price = car.find_element(By.CSS_SELECTOR, ".Dprice").text
    except:
        old_price = ""

    try:
        savings = car.find_element(By.CSS_SELECTOR, ".SavingsGradientBg").text
    except:
        savings = ""

    try:
        location = car.find_element(By.CSS_SELECTOR, ".distanceText").text
    except:
        location = ""

    try:
        image = car.find_element(By.CSS_SELECTOR, ".image_container img").get_attribute("src")
    except:
        image = ""

    try:
        link = car.find_element(By.CSS_SELECTOR, "h3.title a").get_attribute("href")
    except:
        link = ""

    data.append({
        "Title": title,
        "KM Driven": kms,
        "Fuel": fuel,
        "Transmission": transmission,
        "Price": price,
        "Old Price": old_price,
        "Savings": savings,
        "Location": location,
        "Image": image,
        "Car Link": link
    })

driver.quit()

df = pd.DataFrame(data)
df.to_csv("cardekho_used_cars.csv", index=False)

print("Scraping Complete! File saved as cardekho_used_cars.csv")


In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
import pandas as pd
import time

options = Options()
options.add_argument("--start-maximized")
options.add_argument("user-agent=Mozilla/5.0")
driver = webdriver.Chrome(options=options)

cities = [
    "delhi", "mumbai", "pune", "bangalore", "hyderabad",
    "chennai", "kolkata", "ahmedabad", "jaipur", "lucknow",
    "surat", "nagpur", "indore", "bhopal", "coimbatore",
    "vadodara", "rajkot", "kochi", "patna", "chandigarh"
]

final_data = []

for city in cities:
    print(f"\n🔵 Scraping city: {city}")
    
    for page in range(1, 40):   # प्रत्येक city चे 40 pages
        url = f"https://www.cardekho.com/used-cars+in+{city}-page{page}.htm"
        driver.get(url)
        time.sleep(3)

        cars = driver.find_elements(By.CSS_SELECTOR, "div.NewUcExCard")
        print(f"Page {page}: {len(cars)} cars")

        if len(cars) == 0:
            break

        for car in cars:
            try: title = car.find_element(By.CSS_SELECTOR, "h3.title a").text
            except: title = ""

            try: details = car.find_element(By.CSS_SELECTOR, ".dotsDetails").text
            except: details = ""

            try: price = car.find_element(By.CSS_SELECTOR, ".Price p").text
            except: price = ""

            try: old_price = car.find_element(By.CSS_SELECTOR, ".Dprice").text
            except: old_price = ""

            try: savings = car.find_element(By.CSS_SELECTOR, ".SavingsGradientBg").text
            except: savings = ""

            try: image = car.find_element(By.CSS_SELECTOR, ".image_container img").get_attribute("src")
            except: image = ""

            try: link = car.find_element(By.CSS_SELECTOR, "h3.title a").get_attribute("href")
            except: link = ""

            final_data.append({
                "City": city,
                "Title": title,
                "Details": details,
                "Price": price,
                "Old Price": old_price,
                "Savings": savings,
                "Image": image,
                "Link": link
            })

df = pd.DataFrame(final_data)
df.to_csv("CarDekho_India_UsedCars.csv", index=False)

driver.quit()

print("\n🎉 DONE! Full India dataset saved as CarDekho_India_UsedCars.csv")
print("Total rows:", len(df))



🔵 Scraping city: delhi
Page 1: 20 cars
Page 2: 20 cars
Page 3: 20 cars
Page 4: 20 cars
Page 5: 20 cars
Page 6: 20 cars
Page 7: 20 cars
Page 8: 20 cars
Page 9: 20 cars
Page 10: 20 cars
Page 11: 20 cars
Page 12: 20 cars
Page 13: 20 cars
Page 14: 20 cars
Page 15: 20 cars
Page 16: 20 cars
Page 17: 20 cars
Page 18: 20 cars
Page 19: 20 cars
Page 20: 20 cars
Page 21: 20 cars
Page 22: 20 cars
Page 23: 20 cars
Page 24: 20 cars
Page 25: 20 cars
Page 26: 20 cars
Page 27: 20 cars
Page 28: 20 cars
Page 29: 20 cars
Page 30: 20 cars
Page 31: 20 cars
Page 32: 20 cars
Page 33: 20 cars
Page 34: 20 cars
Page 35: 20 cars
Page 36: 20 cars
Page 37: 20 cars
Page 38: 20 cars
Page 39: 20 cars

🔵 Scraping city: mumbai
Page 1: 20 cars
Page 2: 20 cars
Page 3: 20 cars
Page 4: 20 cars
Page 5: 20 cars
Page 6: 20 cars
Page 7: 20 cars
Page 8: 20 cars
Page 9: 20 cars
Page 10: 20 cars
Page 11: 20 cars
Page 12: 20 cars
Page 13: 20 cars
Page 14: 20 cars
Page 15: 20 cars
Page 16: 20 cars
Page 17: 20 cars
Page 18: 20 cars


In [3]:
df.to_csv("CarDekho_India_UsedCars.csv", index=False, sep = ",")

In [4]:
df

,City,Title,Details,Price,Old Price,Savings,Image,Link
0,delhi,2024 Kia Sonet HTK Plus,"10,000 kms • Petrol • Manual",₹8.40 Lakh,,,https://images10.gaadi.com/usedcar_image/48612...,https://www.cardekho.com/used-car-details/used...
1,delhi,2022 Renault Kiger RXZ,"50,214 kms • Petrol • Manual",₹5.75 Lakh,,,https://images10.gaadi.com/usedcar_image/49351...,https://www.cardekho.com/buy-used-car-details/...
2,delhi,2024 Nissan Magnite XV,"19,000 kms • Petrol • Manual",₹6.80 Lakh,,,https://images10.gaadi.com/usedcar_image/49251...,https://www.cardekho.com/used-car-details/used...
3,delhi,2022 Renault Kiger RXT Opt,"14,464 kms • Petrol • Manual",₹5.50 Lakh,,,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/buy-used-car-details/...
4,delhi,2022 Kia Sonet HTX Turbo iMT BSVI,"37,741 kms • Petrol • Manual",₹8 Lakh,,,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...
...,...,...,...,...,...,...,...,...
15595,chandigarh,2023 Maruti Alto K10 VXI,"28,199 kms • Petrol • Manual",₹3.80 Lakh,₹4.19 Lakh,"(Save ₹39,131)",https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/buy-used-car-details/...
15596,chandigarh,2024 MG Astor Select CVT,"20,000 kms • Petrol • Automatic",₹12.50 Lakh,,,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...
15597,chandigarh,2024 Mahindra XUV 3XO MX3,"8,000 kms • Petrol • Manual",₹8.50 Lakh,,,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...
15598,chandigarh,2024 Kia Sonet HTX Turbo iMT,"17,000 kms • Petrol • Manual",₹10.70 Lakh,,,https://stimg.cardekho.com/pwa/img/spacer3x2.png,https://www.cardekho.com/used-car-details/used...
